# Week 15: Graph Neural Networks & Geometric Deep Learning
## Notebook 10: Dynamic Graph CNN (DGCNN) & EdgeConv on 3D Point Clouds

### 1. Conceptual Intuition & Theoretical Background
3D point clouds $\mathcal{P} = \{x_1, \dots, x_n\} \subset \mathbb{R}^3$ obtained from LiDAR or RGB-D sensors are irregular and unordered. PointNet evaluated points independently with shared MLPs, missing local geometric neighborhood relationships.

**EdgeConv (Wang et al., 2019):**
Constructs a dynamic k-Nearest Neighbor graph in **feature space** and applies edge convolutions:
$$e_{ij} = h_\Theta(x_i,\ x_j - x_i)$$
Where:
- $x_i$: Central node feature (capturing global coordinates).
- $x_j - x_i$: Relative displacement vector (capturing local geometric structure).

**Aggregated EdgeConv Representation:**
$$x_i^{(l+1)} = \max_{j \in \mathcal{N}(i)} \text{ReLU}\left( \Theta_1 x_i + \Theta_2 (x_j - x_i) \right)$$

**Dynamic Graph Construction:**
The graph is **recalculated at every layer** in the newly projected latent feature space, grouping semantically related points together even if they are physically far apart in raw Euclidean space.

---

### 2. Scratch Python Implementation: EdgeConv Layer

In [ ]:
import numpy as np

class EdgeConvLayerScratch:
    def __init__(self, in_channels=3, out_channels=8, k_neighbors=3):
        self.k = k_neighbors
        self.in_ch = in_channels
        self.out_ch = out_channels
        
        # Linear projection weights: Theta_1 (central) and Theta_2 (relative)
        np.random.seed(42)
        self.theta1 = np.random.normal(0, 0.1, (in_channels, out_channels))
        self.theta2 = np.random.normal(0, 0.1, (in_channels, out_channels))
        
    def forward(self, points):
        """
        points: shape (n_points, in_channels)
        """
        N, C = points.shape
        # 1. Compute pairwise Euclidean distances in current feature space
        diff = points[:, np.newaxis, :] - points[np.newaxis, :, :]
        dist_sq = np.sum(diff ** 2, axis=-1)
        
        # 2. Extract k-nearest neighbors for each point
        knn_indices = np.argsort(dist_sq, axis=1)[:, 1 : self.k + 1] # Skip self (index 0)
        
        # 3. Compute edge features: e_ij = Theta1 * x_i + Theta2 * (x_j - x_i)
        edge_features = np.zeros((N, self.k, self.out_ch))
        
        for i in range(N):
            x_i = points[i]
            for idx_k, j in enumerate(knn_indices[i]):
                x_j = points[j]
                rel_disp = x_j - x_i
                
                # Edge linear combination
                e_ij = np.dot(x_i, self.theta1) + np.dot(rel_disp, self.theta2)
                edge_features[i, idx_k] = np.maximum(0, e_ij) # ReLU
                
        # 4. Neighborhood Aggregation via Channel-Wise Max Pooling
        out_features = np.max(edge_features, axis=1) # (N, out_channels)
        return out_features

# Simulate 3D Point Cloud (Cube vertices)
point_cloud = np.array([
    [0.0, 0.0, 0.0],
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [1.0, 1.0, 0.0],
    [0.5, 0.5, 1.0]
])

edgeconv = EdgeConvLayerScratch(in_channels=3, out_channels=6, k_neighbors=2)
out_repr = edgeconv.forward(point_cloud)

print("--- Dynamic Graph CNN (EdgeConv) ---")
print(f"Input Point Cloud Coordinates: {point_cloud.shape}")
print(f"Output Feature Dimension:       {out_repr.shape}")
print("Local displacement features capture fine 3D surface geometry!")

### 3. Step-by-Step Code Walkthrough
1. **Dynamic k-NN Graph:** Computes neighbors dynamically in feature space at each layer.
2. **Relative Displacement ($x_j - x_i$):** Captures local surface normals and curvature.
3. **Max-Pooling:** Provides permutation invariance with respect to neighbor ordering.

---

### 4. Industry Applications & Trade-offs
- **LiDAR Point Cloud Segmentation (Waymo, Tesla):** Classifying 3D vehicles, pedestrians, and road barriers.
- **Trade-off:** Recalculating all-pairs distances for dynamic k-NN graphs at every layer introduces computational overhead on large point clouds.

---

### 5. Top Interview Questions & Answers
**Q1: Why does EdgeConv include both $x_i$ and $(x_j - x_i)$ rather than just neighbor features $x_j$?**
> **Answer:** Including $x_i$ preserves the global spatial coordinates of the point, while $(x_j - x_i)$ captures local translation-invariant geometric features (like surface orientation). Combining them provides both global context and fine local detail.

**Q2: What is the benefit of dynamic graph construction over a fixed graph in 3D vision?**
> **Answer:** In deep layers, points that belong to the same semantic class (e.g., all 4 wheels of a car) move close together in feature space, allowing EdgeConv to aggregate information across spatially distant but semantically related components.